#### Capstone Project

**THE PRICE IS RIGHT**

1. Data Curation

-> 2. **Data Pre-Processing**

3. Evaluation against Baselines and Traditional ML

4. Deep Learning and LLMs

5. Fine-Tuning a Frontier Model

In [1]:
# to move one folder up for pricer folder

import os
import sys

sys.path.insert(0,os.path.abspath('..'))

In [2]:
# imports

from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

In [3]:
# selecting the dataset

LITE_MODE = True

In [4]:
# loading the dataset

user_name = 'lalam0'

dataset = f"{user_name}/items_raw_lite" if LITE_MODE else f"{user_name}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f'Loaded {len(items):,} items')

Loaded 22,000 items


In [5]:
# sample datapoint

items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [6]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [7]:
# give every item an ID

for index, item in enumerate(items):
    item.id = index

In [8]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [9]:
# system prompt

SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [10]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [11]:
# calling gpt-oss-20b to pre-process the first element in the dataset

messages = [{'role':'system', 'content':SYSTEM_PROMPT}, {'role':'user', 'content':items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f'Input Tokens: {response.usage.prompt_tokens}')
print(f'Ouptut Tokens: {response.usage.completion_tokens}')
print(f'Cost: {response._hidden_params['response_cost']*100:.3f} cents')

Title: Schlage F59 & 613 Andover Interior Half Knob with Deadbolt – Oil Rubbed Bronze  
Category: Home Hardware  
Brand: Schlage  
Description: A secure, easy‑to‑install interior half knob featuring an integrated deadbolt in classic oil‑rubbed bronze.  
Details: Made from solid metal with a lifetime mechanical and finish warranty, the knob requires a 4" center‑to‑center door prep and is compatible with the F58 handle set.

Input Tokens: 446
Ouptut Tokens: 114
Cost: 0.007 cents


In [13]:
# calling llama3.2 via ollama

messages = [{'role':'system', 'content':SYSTEM_PROMPT}, {'role':'user', 'content':items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", base_url="http://localhost:11434")

print(response.choices[0].message.content)
print()
print(f'Input Tokens: {response.usage.prompt_tokens}')
print(f'Ouptut Tokens: {response.usage.completion_tokens}')
print(f'Cost: {response._hidden_params['response_cost']*100:.3f} cents')

### Product Description

### Title: Schlage Front Door Knob Set

### Category: Home Security

### Brand: Schlage

### Description: A durable oil rubbed bronze knob set with a deadbolt for enhanced security and peace of mind.

### Details: Features easy-to-install design for a secure and convenient locking system.

Input Tokens: 406
Ouptut Tokens: 66
Cost: 0.000 cents


In [14]:
# model = llama3.2

MODEL = "openai/gpt-oss-20b"

In [15]:
# to convert the datapoint into jsonline for batch processing

def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}]}
    line = {"custom_id": str(item.id), "method":"POST", "url":"/v1/chat/completions", "body":body}
    return json.dumps(line)


In [16]:
# original datapoint

items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [17]:
# after converting the datapoint into jsonl

make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece m

In [18]:
# creating a jsonl file

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end+1):
            f.write(make_jsonl(items[i]))
            f.write('\n')

In [19]:
# to create the first jsonl file

make_file(0, 100, "../jsonl/0_100.jsonl")

In [ ]:


import os
from openai import OpenAI

groq_api_key = os.getenv('GROQ_API_KEY')
groq_base_url = os.getenv('GROQ_BASE_URL')

groq = OpenAI(api_key=groq_api_key, base_url=groq_base_url)

In [ ]:
with open("../jsonl/0_100.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
# to download the final dataset (after pre-processing) from ed's huggingface
# small dataset

user_name = "ed-donner"
lite = f"{user_name}/items_lite"

train, val, test = Item.from_hub(lite)

items = train + val + test

print(f'Loaded {len(items)} items')

Loaded 22000 items


In [ ]:
# verifying a sample datapoint

for item in items[0]:
    print(item)

('title', 'Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)')
('category', 'Tools_and_Home_Improvement')
('price', 64.3)
('full', None)
('weight', 1.5)
('summary', 'Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.')
('prompt', None)
('id', None)


In [30]:
# push the lite dataset (after pre-processing) into my huggingface datasets

user_name = "lalam0"
lite = f"{user_name}/items_lite"

Item.push_to_hub(lite, train, val, test)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [37]:
# downloading large dataset

user_name = "ed-donner"
full = f"{user_name}/items_full"

train, val, test = Item.from_hub(full)

items_full = train + val + test

print(f"Loaded {len(items_full)} items")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  243MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.04MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 820000 items


In [38]:
# to upload the dataset

user_name = "lalam0"
full = f"{user_name}/items_full"

Item.push_to_hub(full, train, val, test)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/800 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            